# Onda 1: Exploracao da API Dados Abertos da Camara dos Deputados

Objetivo: inspecionar os retornos reais dos 5 grupos de endpoints (`/partidos`, `/deputados`, `/proposicoes`, `/votacoes` e `/deputados/{id}/despesas`), mapear paginacao (`links -> next`), tipos de dados, valores nulos e anomalias antes de congelar o DDL em `sql/01_schema_star.sql`.

In [ ]:
from datetime import date, timedelta
import json
import pandas as pd
import requests

BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"
HEADERS = {"Accept": "application/json"}
hoje = date.today()
inicio_7d = (hoje - timedelta(days=7)).isoformat()
fim = hoje.isoformat()
print(f"Janela piloto: {inicio_7d} ate {fim}")

In [ ]:
session = requests.Session()
partidos = session.get(f"{BASE_URL}/partidos", params={"itens": 100}, headers=HEADERS, timeout=30).json()
deputados = session.get(f"{BASE_URL}/deputados", params={"itens": 100}, headers=HEADERS, timeout=30).json()
proposicoes = session.get(
    f"{BASE_URL}/proposicoes"
    , params={"dataApresentacaoInicio": inicio_7d, "dataApresentacaoFim": fim, "itens": 100}
    , headers=HEADERS
    , timeout=30
).json()
votacoes = session.get(
    f"{BASE_URL}/votacoes"
    , params={"dataInicio": inicio_7d, "dataFim": fim, "itens": 100}
    , headers=HEADERS
    , timeout=30
).json()

for nome, payload in [("partidos", partidos), ("deputados", deputados), ("proposicoes", proposicoes), ("votacoes", votacoes)]:
    df = pd.json_normalize(payload.get("dados", []))
    print(f"\n--- {nome} ({len(df)} itens na pag 1) ---")
    print(df.dtypes)

In [ ]:
dep_id = deputados["dados"][0]["id"]
mes_ref = (hoje.replace(day=1) - timedelta(days=1))
despesas = session.get(
    f"{BASE_URL}/deputados/{dep_id}/despesas"
    , params={"ano": mes_ref.year, "mes": mes_ref.month, "itens": 100}
    , headers=HEADERS
    , timeout=30
).json()
df_desp = pd.json_normalize(despesas.get("dados", []))
print(f"Deputado {dep_id} | {mes_ref.month}/{mes_ref.year} | registros: {len(df_desp)}")
if not df_desp.empty:
    print("Negativos em valorLiquido:", int((df_desp["valorLiquido"] < 0).sum()))
    display(df_desp.head(5))

## Achados reais da exploracao (Onda 1)

1. **Paginacao:** todos os endpoints retornam `links` com `rel == 'next'` e suportam `itens=100`.
2. **`proposicoes`:** emendas e substitutivos (`SBT`, `EMC`, `PRL`) frequentemente chegam com `ano = 0`, e `dataApresentacao` vem no formato `YYYY-MM-DDTHH:MM`.
3. **`votacoes`:** o `id` e alfanumerico (ex.: `'949156-136'`), `aprovacao` pode ser `0`, `1` ou nulo, e `uriProposicaoObjeto` pode apontar para proposicoes antigas fora da janela de 30 dias (logo `fato_votacoes.id_proposicao` nao deve bloquear carga por FK estrita).
4. **`despesas`:** `codDocumento` chega como string, um mesmo documento pode ter multiplas `parcela`, e a API retorna registros com `valorLiquido < 0` (ajustes/estornos da CEAP) que devem ser capturados e filtrados na camada de qualidade antes da carga.